In [ ]:
!pip -q install apify-client

In [ ]:
from getpass import getpass
from apify_client import ApifyClient

APIFY_TOKEN = getpass("Enter your Apify API token: ")

client = ApifyClient(APIFY_TOKEN)

print("Apify client initialized.")

In [ ]:
try:
    account = client.user().get()

    if account:
        print("Apify connection successful!")
        print("Authenticated account:", account.get("username", "Connected"))
    else:
        print("No account details returned. Check your token.")

except Exception as e:
    print("Connection failed:", str(e))

In [ ]:
try:
    account = client.user().get()

    if account:
        print("Apify connection successful!")
        print("Account details:", account)
    else:
        print("No account details returned.")

except Exception as e:
    print("Error:", str(e))

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

tweets = pd.read_csv(filename)

print("Rows and columns:", tweets.shape)
print("Available columns:", tweets.columns.tolist())
display(tweets.head(3))

In [ ]:
import pandas as pd

required = ["text", "createdAt", "likeCount", "retweetCount", "replyCount", "bookmarkCount"]
available = [col for col in required if col in tweets.columns]

clean_tweets = tweets[available].copy()

clean_tweets["text"] = clean_tweets["text"].fillna("").astype(str).str.strip()
clean_tweets["createdAt"] = pd.to_datetime(
    clean_tweets["createdAt"], errors="coerce", utc=True
)

clean_tweets = clean_tweets[
    (clean_tweets["text"] != "") & clean_tweets["createdAt"].notna()
].drop_duplicates(subset=["text", "createdAt"])

for col in ["likeCount", "retweetCount", "replyCount", "bookmarkCount"]:
    if col in clean_tweets.columns:
        clean_tweets[col] = pd.to_numeric(
            clean_tweets[col], errors="coerce"
        ).fillna(0)

clean_tweets = clean_tweets.sort_values("createdAt").reset_index(drop=True)
clean_tweets.to_csv("clean_tweets.csv", index=False)

print("Clean dataset shape:", clean_tweets.shape)
print("Date range:", clean_tweets["createdAt"].min(), "to", clean_tweets["createdAt"].max())
display(clean_tweets[["createdAt", "text"]].head())

In [ ]:
!pip -q install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

clean_tweets["sentiment"] = clean_tweets["text"].apply(
    lambda text: analyzer.polarity_scores(text)["compound"]
)

clean_tweets["sentiment_label"] = clean_tweets["sentiment"].apply(
    lambda score: "Positive" if score >= 0.05
    else "Negative" if score <= -0.05
    else "Neutral"
)

display(clean_tweets[["createdAt", "text", "sentiment", "sentiment_label"]])
print(clean_tweets["sentiment_label"].value_counts())

In [ ]:
!pip -q install yfinance

import yfinance as yf
import pandas as pd

prices = yf.download(
    "NVDA",
    period="6mo",
    interval="1d",
    auto_adjust=True,
    progress=False
)

if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)

prices = prices.reset_index()
prices.columns = [str(col).lower().replace(" ", "_") for col in prices.columns]
prices = prices.rename(columns={"date": "date", "close": "close", "volume": "volume"})

prices["date"] = pd.to_datetime(prices["date"], utc=True).dt.date
prices = prices.sort_values("date").reset_index(drop=True)

print("Price data shape:", prices.shape)
print("Date range:", prices["date"].min(), "to", prices["date"].max())
display(prices.tail())

In [ ]:
clean_tweets["date"] = pd.to_datetime(clean_tweets["createdAt"], utc=True).dt.date

daily_sentiment = clean_tweets.groupby("date").agg(
    sentiment_mean=("sentiment", "mean"),
    sentiment_std=("sentiment", "std"),
    tweet_count=("sentiment", "size"),
    positive_count=("sentiment_label", lambda s: (s == "Positive").sum()),
    negative_count=("sentiment_label", lambda s: (s == "Negative").sum())
).reset_index()

dataset = prices.merge(daily_sentiment, on="date", how="left")

dataset["sentiment_available"] = dataset["tweet_count"].notna().astype(int)
dataset["tweet_count"] = dataset["tweet_count"].fillna(0)
dataset["sentiment_mean"] = dataset["sentiment_mean"].fillna(0)
dataset["sentiment_std"] = dataset["sentiment_std"].fillna(0)
dataset["positive_count"] = dataset["positive_count"].fillna(0)
dataset["negative_count"] = dataset["negative_count"].fillna(0)

print("Combined dataset shape:", dataset.shape)
print("Days with tweets:", dataset["sentiment_available"].sum())
display(dataset.tail(10))

In [ ]:
import numpy as np
import pandas as pd

model_data = prices.copy()
model_data["return_1d"] = model_data["close"].pct_change()
model_data["return_5d"] = model_data["close"].pct_change(5)
model_data["volatility_5d"] = model_data["return_1d"].rolling(5).std()
model_data["volatility_10d"] = model_data["return_1d"].rolling(10).std()
model_data["volume_change"] = model_data["volume"].pct_change()

for lag in [1, 2, 3, 5]:
    model_data[f"close_lag_{lag}"] = model_data["close"].shift(lag)

for horizon in range(6):
    model_data[f"target_n{horizon}"] = model_data["close"].shift(-horizon)

model_data = model_data.replace([np.inf, -np.inf], np.nan)

print("Dataset shape:", model_data.shape)
display(model_data.tail(10))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd

feature_cols = [
    "open", "high", "low", "close", "volume",
    "return_1d", "return_5d", "volatility_5d",
    "volatility_10d", "volume_change",
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_5"
]

target_col = "target_n1"

data = model_data.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

X = data[feature_cols]
y = data[target_col]

tscv = TimeSeriesSplit(n_splits=5)
results = []
predictions = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    model_pred = model.predict(X_test)
    baseline_pred = data["close"].iloc[test_idx].to_numpy()

    results.append({
        "fold": fold,
        "model_MAE": mean_absolute_error(y_test, model_pred),
        "model_RMSE": np.sqrt(mean_squared_error(y_test, model_pred)),
        "baseline_MAE": mean_absolute_error(y_test, baseline_pred),
        "baseline_RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred))
    })

    predictions.extend([
        {
            "date": data["date"].iloc[i],
            "actual": y.iloc[j],
            "model_prediction": model_pred[k],
            "baseline_prediction": baseline_pred[k]
        }
        for k, (i, j) in enumerate(zip(test_idx, test_idx))
    ])

results_df = pd.DataFrame(results)
predictions_df = pd.DataFrame(predictions)

display(results_df.round(4))
print("Mean model MAE:", round(results_df["model_MAE"].mean(), 4))
print("Mean baseline MAE:", round(results_df["baseline_MAE"].mean(), 4))
display(predictions_df.tail())

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd

feature_cols = [
    "open", "high", "low", "close", "volume",
    "return_1d", "return_5d", "volatility_5d",
    "volatility_10d", "volume_change",
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_5"
]

horizon_results = []
horizon_predictions = {}

for horizon in range(6):
    target_col = f"target_n{horizon}"
    data_h = model_data.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

    X = data_h[feature_cols]
    y = data_h[target_col]
    cv = TimeSeriesSplit(n_splits=5)

    actual_all = []
    model_all = []
    baseline_all = []

    for train_idx, test_idx in cv.split(X):
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=3,
            random_state=42
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        model_pred = model.predict(X.iloc[test_idx])

        actual_all.extend(y.iloc[test_idx].to_numpy())
        model_all.extend(model_pred)
        baseline_all.extend(data_h["close"].iloc[test_idx].to_numpy())

    horizon_results.append({
        "horizon": f"n+{horizon}",
        "model_MAE": mean_absolute_error(actual_all, model_all),
        "model_RMSE": np.sqrt(mean_squared_error(actual_all, model_all)),
        "baseline_MAE": mean_absolute_error(actual_all, baseline_all),
        "baseline_RMSE": np.sqrt(mean_squared_error(actual_all, baseline_all))
    })

    horizon_predictions[f"n+{horizon}"] = {
        "actual": actual_all,
        "model": model_all,
        "baseline": baseline_all
    }

horizon_results_df = pd.DataFrame(horizon_results)
display(horizon_results_df.round(4))

In [ ]:
from getpass import getpass

TAVILY_API_KEY = getpass("Enter your Tavily API key: ")

In [ ]:
!pip -q install tavily-python

from tavily import TavilyClient
import pandas as pd

tavily = TavilyClient(api_key=TAVILY_API_KEY)

response = tavily.search(
    query="NVIDIA NVDA stock earnings data center AI business news",
    topic="news",
    days=30,
    max_results=20,
    include_answer=True
)

news_data = pd.DataFrame(response.get("results", []))

print("News articles collected:", len(news_data))
display(news_data.head())

In [ ]:
!pip -q install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

analyzer = SentimentIntensityAnalyzer()

news_data["published_at"] = pd.to_datetime(
    news_data["published_date"], errors="coerce", utc=True
)

news_data["news_text"] = (
    news_data["title"].fillna("") + ". " +
    news_data["content"].fillna("")
).str.strip()

news_data["news_sentiment"] = news_data["news_text"].apply(
    lambda text: analyzer.polarity_scores(text)["compound"]
)

news_data["sentiment_label"] = news_data["news_sentiment"].apply(
    lambda score: "Positive" if score >= 0.05
    else "Negative" if score <= -0.05
    else "Neutral"
)

news_data["date"] = news_data["published_at"].dt.date

display(
    news_data[
        ["published_at", "title", "news_sentiment", "sentiment_label", "url"]
    ]
)

print("\nSentiment distribution:")
print(news_data["sentiment_label"].value_counts())

print("\nArticles with publication dates:", news_data["date"].notna().sum())

In [ ]:
news_daily = (
    news_data.dropna(subset=["date"])
    .groupby("date")
    .agg(
        news_sentiment_mean=("news_sentiment", "mean"),
        news_article_count=("news_sentiment", "size"),
        news_positive_count=("sentiment_label", lambda s: (s == "Positive").sum()),
        news_negative_count=("sentiment_label", lambda s: (s == "Negative").sum())
    )
    .reset_index()
)

news_daily["date"] = pd.to_datetime(news_daily["date"]).dt.date

print("Dated news records:", len(news_daily))
display(news_daily.sort_values("date"))

In [ ]:
combined_data = prices.merge(news_daily, on="date", how="left")

combined_data["news_available"] = combined_data["news_article_count"].notna().astype(int)

for col in [
    "news_sentiment_mean",
    "news_article_count",
    "news_positive_count",
    "news_negative_count"
]:
    combined_data[col] = combined_data[col].fillna(0)

print("Combined dataset shape:", combined_data.shape)
print("Days with dated news:", combined_data["news_available"].sum())
display(combined_data.tail())

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def process_tweets(tweets_df):
    df = tweets_df.copy()
    df["createdAt"] = pd.to_datetime(df["createdAt"], errors="coerce", utc=True)
    df = df.dropna(subset=["createdAt", "text"])
    df["date"] = df["createdAt"].dt.date
    df["sentiment"] = df["text"].apply(
        lambda text: analyzer.polarity_scores(str(text))["compound"]
    )
    df["sentiment_label"] = df["sentiment"].apply(
        lambda score: "Positive" if score >= 0.05
        else "Negative" if score <= -0.05
        else "Neutral"
    )
    return df

def aggregate_tweet_sentiment(tweets_df):
    return tweets_df.groupby("date").agg(
        x_sentiment_mean=("sentiment", "mean"),
        x_sentiment_std=("sentiment", "std"),
        x_tweet_count=("sentiment", "size"),
        x_positive_count=("sentiment_label", lambda s: (s == "Positive").sum()),
        x_negative_count=("sentiment_label", lambda s: (s == "Negative").sum())
    ).reset_index()

In [ ]:
tweets_raw = pd.read_csv("/content/your_tweet_dataset.csv")

processed_tweets = process_tweets(tweets_raw)
x_daily = aggregate_tweet_sentiment(processed_tweets)

combined_data = prices.merge(news_daily, on="date", how="left")
combined_data = combined_data.merge(x_daily, on="date", how="left")

print("Combined dataset shape:", combined_data.shape)
print("Dates with X sentiment:", combined_data["x_tweet_count"].notna().sum())
print("Dates with news:", combined_data["news_article_count"].notna().sum())

display(combined_data.tail())

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import glob
import os

csv_files = glob.glob("/content/*.csv")

tweets_list = []

for file in csv_files:
    df = pd.read_csv(file)
    tweets_list.append(df)

tweets = pd.concat(tweets_list, ignore_index=True)

tweets = tweets.drop_duplicates(subset="id")
tweets = tweets.dropna(subset=["text"])

tweets["createdAt"] = pd.to_datetime(
    tweets["createdAt"], errors="coerce", utc=True
)

tweets = tweets.sort_values("createdAt").reset_index(drop=True)

print("Total unique tweets:", len(tweets))
print("Date range:", tweets["createdAt"].min(), "to", tweets["createdAt"].max())

tweets.head()

In [ ]:
!pip -q install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

tweets["sentiment"] = tweets["text"].apply(
    lambda text: analyzer.polarity_scores(str(text))["compound"]
)

tweets["sentiment_label"] = tweets["sentiment"].apply(
    lambda score: "Positive" if score >= 0.05
    else "Negative" if score <= -0.05
    else "Neutral"
)

tweets[["createdAt", "text", "sentiment", "sentiment_label"]].head()

In [ ]:
print(tweets["sentiment_label"].value_counts())
print("Tweets per date:")
print(tweets.groupby(tweets["createdAt"].dt.date).size())

In [ ]:
tweets["date"] = tweets["createdAt"].dt.date

daily_sentiment = (
    tweets.groupby("date")
    .agg(
        tweet_count=("sentiment", "size"),
        sentiment_mean=("sentiment", "mean"),
        sentiment_median=("sentiment", "median"),
        positive_count=("sentiment_label", lambda s: (s == "Positive").sum()),
        neutral_count=("sentiment_label", lambda s: (s == "Neutral").sum()),
        negative_count=("sentiment_label", lambda s: (s == "Negative").sum()),
    )
    .reset_index()
)

daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"]).dt.date
prices["date"] = pd.to_datetime(prices["date"]).dt.date

combined_data = prices.merge(daily_sentiment, on="date", how="left")

combined_data["tweet_count"] = combined_data["tweet_count"].fillna(0).astype(int)

combined_data.head()

In [ ]:
print("Trading days:", len(combined_data))
print("Trading days with tweets:", (combined_data["tweet_count"] > 0).sum())
print("Total tweets merged:", combined_data["tweet_count"].sum())

In [ ]:
combined_data["has_tweets"] = (combined_data["tweet_count"] > 0).astype(int)

combined_data["sentiment_mean"] = combined_data["sentiment_mean"].fillna(0)
combined_data["sentiment_median"] = combined_data["sentiment_median"].fillna(0)

combined_data["sentiment_available"] = combined_data["has_tweets"]

print("Rows:", len(combined_data))
print("Rows with sentiment:", combined_data["sentiment_available"].sum())
print("Sentiment coverage:", round(combined_data["sentiment_available"].mean() * 100, 2), "%")

combined_data[combined_data["has_tweets"] == 1][
    ["date", "close", "tweet_count", "sentiment_mean", "positive_count", "neutral_count", "negative_count"]
]

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

forecast_data = combined_data.copy().sort_values("date").reset_index(drop=True)

forecast_data["return_1d"] = forecast_data["close"].pct_change()
forecast_data["return_5d"] = forecast_data["close"].pct_change(5)
forecast_data["volatility_5d"] = forecast_data["return_1d"].rolling(5).std()
forecast_data["volume_change"] = forecast_data["volume"].pct_change()

for lag in [1, 2, 3, 5]:
    forecast_data[f"close_lag_{lag}"] = forecast_data["close"].shift(lag)

for horizon in range(1, 6):
    forecast_data[f"target_n{horizon}"] = forecast_data["close"].shift(-horizon)

forecast_data = forecast_data.replace([np.inf, -np.inf], np.nan)

forecast_data[["date", "close", "sentiment_mean", "tweet_count", "target_n1", "target_n5"]].tail(10)

In [ ]:
price_features = [
    "open", "high", "low", "close", "volume",
    "return_1d", "return_5d", "volatility_5d",
    "volume_change", "close_lag_1", "close_lag_2",
    "close_lag_3", "close_lag_5"
]

sentiment_features = price_features + [
    "sentiment_mean", "sentiment_median",
    "tweet_count", "sentiment_available"
]

results = []

for name, features in [
    ("Price only", price_features),
    ("Price + sentiment", sentiment_features)
]:
    data = forecast_data.dropna(subset=features + ["target_n1"]).copy()
    X = data[features]
    y = data["target_n1"]

    splitter = TimeSeriesSplit(n_splits=5)
    model_errors = []
    baseline_errors = []

    for train_idx, test_idx in splitter.split(X):
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=3,
            random_state=42
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[test_idx])

        model_errors.extend(abs(y.iloc[test_idx].to_numpy() - predictions))
        baseline_errors.extend(
            abs(y.iloc[test_idx].to_numpy() - data["close"].iloc[test_idx].to_numpy())
        )

    results.append({
        "Model": name,
        "MAE": np.mean(model_errors),
        "Baseline MAE": np.mean(baseline_errors)
    })

pd.DataFrame(results).round(4)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

features = price_features + [
    "sentiment_mean", "sentiment_median",
    "tweet_count", "sentiment_available"
]

data = forecast_data.dropna(subset=features + ["target_n1"]).copy()
X = data[features]
y = data["target_n1"]

predictions = []

splitter = TimeSeriesSplit(n_splits=5)

for train_idx, test_idx in splitter.split(X):
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3,
        random_state=42
    )
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    fold = data.iloc[test_idx][["date", "close"]].copy()
    fold["actual_next_close"] = y.iloc[test_idx].to_numpy()
    fold["predicted_next_close"] = model.predict(X.iloc[test_idx])
    predictions.append(fold)

oos = pd.concat(predictions).sort_values("date").reset_index(drop=True)

oos["actual_return"] = (
    oos["actual_next_close"] / oos["close"] - 1
)

oos["strategy_return"] = np.where(
    oos["predicted_next_close"] > oos["close"],
    oos["actual_return"],
    0
)

oos["equity"] = (1 + oos["strategy_return"]).cumprod()
oos["running_peak"] = oos["equity"].cummax()
oos["drawdown"] = oos["equity"] / oos["running_peak"] - 1

daily_returns = oos["strategy_return"]
downside_returns = daily_returns[daily_returns < 0]

sharpe = (
    daily_returns.mean() / daily_returns.std(ddof=1) * np.sqrt(252)
    if daily_returns.std(ddof=1) > 0 else np.nan
)

sortino = (
    daily_returns.mean() / np.sqrt((np.minimum(daily_returns, 0) ** 2).mean()) * np.sqrt(252)
    if (np.minimum(daily_returns, 0) ** 2).mean() > 0 else np.nan
)

metrics = pd.DataFrame({
    "Metric": ["Out-of-sample observations", "Sharpe ratio", "Sortino ratio", "Maximum drawdown", "Strategy cumulative return"],
    "Value": [
        len(oos),
        sharpe,
        sortino,
        oos["drawdown"].min(),
        oos["equity"].iloc[-1] - 1
    ]
})

metrics

In [ ]:
print("First and last out-of-sample dates:")
print(oos["date"].min(), oos["date"].max())

print("\nPrediction count:")
print(len(oos))

print("\nActual market return:")
print((oos["actual_next_close"].iloc[-1] / oos["close"].iloc[0]) - 1)

print("\nStrategy return distribution:")
print(oos["strategy_return"].describe())

print("\nNumber of trading days invested:")
print((oos["strategy_return"] != 0).sum())

print("\nFirst 10 predictions:")
display(oos.head(10))

In [ ]:
oos = oos.sort_values("date").reset_index(drop=True)

oos["actual_return"] = (
    oos["actual_next_close"] / oos["close"] - 1
)

oos["position"] = (
    oos["predicted_next_close"] > oos["close"]
).astype(int)

oos["strategy_return"] = oos["position"] * oos["actual_return"]

oos["equity"] = (1 + oos["strategy_return"]).cumprod()
oos["running_peak"] = oos["equity"].cummax()
oos["drawdown"] = oos["equity"] / oos["running_peak"] - 1

daily_returns = oos["strategy_return"]
downside = np.minimum(daily_returns, 0)

sharpe = (
    daily_returns.mean() / daily_returns.std(ddof=1) * np.sqrt(252)
    if daily_returns.std(ddof=1) > 0 else np.nan
)

sortino = (
    daily_returns.mean() / np.sqrt(np.mean(downside ** 2)) * np.sqrt(252)
    if np.mean(downside ** 2) > 0 else np.nan
)

print("Cumulative return:", round(oos["equity"].iloc[-1] - 1, 4))
print("Sharpe ratio:", round(sharpe, 4))
print("Sortino ratio:", round(sortino, 4))
print("Maximum drawdown:", round(oos["drawdown"].min(), 4))
print("Days invested:", int(oos["position"].sum()))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np

forecast_features = price_features + [
    "sentiment_mean",
    "sentiment_median",
    "tweet_count",
    "sentiment_available"
]

latest = forecast_data.iloc[[-1]].copy()

train_data = forecast_data.dropna(
    subset=forecast_features + [f"target_n{i}" for i in range(1, 6)]
).copy()

forecasts = []

for horizon in range(1, 6):
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3,
        random_state=42
    )

    model.fit(
        train_data[forecast_features],
        train_data[f"target_n{horizon}"]
    )

    prediction = model.predict(latest[forecast_features])[0]

    forecasts.append({
        "Horizon": f"n+{horizon}",
        "Forecast Date": (
            pd.to_datetime(latest["date"].iloc[0]) +
            pd.offsets.BDay(horizon)
        ).date(),
        "Predicted Close": round(float(prediction), 2)
    })

forecast_table = pd.DataFrame(
    [{"Horizon": "n+0",
      "Forecast Date": pd.to_datetime(latest["date"].iloc[0]).date(),
      "Predicted Close": round(float(latest["close"].iloc[0]), 2)}]
    + forecasts
)

forecast_table

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.figure(figsize=(12, 6))

plt.plot(
    prices["date"],
    prices["close"],
    label="Historical NVDA Close",
    linewidth=2
)

forecast_dates = pd.to_datetime(forecast_table["Forecast Date"])
forecast_prices = forecast_table["Predicted Close"]

plt.plot(
    forecast_dates,
    forecast_prices,
    marker="o",
    linestyle="--",
    color="orange",
    label="Forecast"
)

plt.axvline(
    pd.to_datetime(forecast_table["Forecast Date"].iloc[0]),
    linestyle=":",
    color="gray"
)

plt.title("NVDA Historical Prices and Forecast (n+0 to n+5)")
plt.xlabel("Date")
plt.ylabel("Closing Price (USD)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
sentiment_plot = tweets.groupby("date").agg(
    tweet_count=("sentiment", "size"),
    sentiment_mean=("sentiment", "mean")
).reset_index()

sentiment_plot["date"] = pd.to_datetime(sentiment_plot["date"])

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(
    sentiment_plot["date"],
    sentiment_plot["tweet_count"],
    width=1.5,
    label="Tweet Count"
)
ax1.set_xlabel("Date")
ax1.set_ylabel("Number of Tweets")

ax2 = ax1.twinx()
ax2.plot(
    sentiment_plot["date"],
    sentiment_plot["sentiment_mean"],
    marker="o",
    color="orange",
    label="Average Sentiment"
)
ax2.set_ylabel("Average Sentiment Score")

plt.title("NVDA Tweet Volume and Sentiment")
fig.tight_layout()
plt.show()

In [ ]:
forecast_table.to_csv("nvda_forecast.csv", index=False)
oos.to_csv("nvda_walk_forward_results.csv", index=False)

print("Forecast and walk-forward results saved.")

In [ ]:
from google.colab import files

files.download("nvda_forecast.csv")
files.download("nvda_walk_forward_results.csv")

In [ ]:
sentiment_plot = tweets.groupby("date").agg(
    tweet_count=("sentiment", "size"),
    sentiment_mean=("sentiment", "mean")
).reset_index()

sentiment_plot["date"] = pd.to_datetime(sentiment_plot["date"])

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(
    sentiment_plot["date"],
    sentiment_plot["tweet_count"],
    width=1.5,
    label="Tweet Count"
)
ax1.set_xlabel("Date")
ax1.set_ylabel("Number of Tweets")

ax2 = ax1.twinx()
ax2.plot(
    sentiment_plot["date"],
    sentiment_plot["sentiment_mean"],
    marker="o",
    color="orange",
    label="Average Sentiment"
)
ax2.set_ylabel("Average Sentiment Score")

plt.title("NVDA Tweet Volume and Sentiment")
fig.tight_layout()
plt.show()

In [ ]:
forecast_table.to_csv("nvda_forecast.csv", index=False)
oos.to_csv("nvda_walk_forward_results.csv", index=False)

print("Files saved successfully.")

In [ ]:
from google.colab import files

files.download("nvda_forecast.csv")
files.download("nvda_walk_forward_results.csv")

In [ ]:
print("MODEL EVALUATION")
print("-" * 35)
print(f"Price-only MAE: 4.7573")
print(f"Price + sentiment MAE: 4.7515")
print(f"Baseline MAE: 4.0350")

print("\nWALK-FORWARD STRATEGY")
print("-" * 35)
print(f"Cumulative return: {(oos['equity'].iloc[-1] - 1) * 100:.2f}%")
print(f"Sharpe ratio: {sharpe:.2f}")
print(f"Sortino ratio: {sortino:.2f}")
print(f"Maximum drawdown: {oos['drawdown'].min() * 100:.2f}%")
print(f"Out-of-sample observations: {len(oos)}")

In [ ]:
import json
import glob
import os

files_found = glob.glob("/content/*.ipynb")

for path in files_found:
    with open(path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))
        output = json.dumps(cell.get("outputs", []))

        terms = ["proxy", "password", "apify"]
        if any(term in (source + output).lower() for term in terms):
            print(f"File: {os.path.basename(path)} | Cell: {i + 1}")
            print("Potential credential-related content found.")

In [1]:

import glob
import json
import os
import re

patterns = {
    "proxy URL with embedded credentials": r"https?://[^/\s:@]+:[^/\s@]+@",
    "possible Apify proxy password field": r"(?i)proxy[_-]?password",
    "possible hardcoded API token": r"(?i)(?:api[_-]?token|api[_-]?key|password)\s*=\s*['\"][^'\"]{6,}['\"]"
}

for path in glob.glob("/content/**/*.ipynb", recursive=True):
    with open(path, encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook.get("cells", [])):
        content = "".join(cell.get("source", []))
        content += json.dumps(cell.get("outputs", []))

        for label, pattern in patterns.items():
            if re.search(pattern, content):
                print(f"{os.path.basename(path)} | Cell {i+1} | {label}")
